# Bollinger Bands & Mean Reversion To Generate Buy/Sell Signals

- **Assets:** Gold (GC=F), AMD, XLE

- **Rules:**
  - Buy when price < lower band
  - Sell when price > moving average

In [9]:
# Imports

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 6)


In [10]:
# Parameters & Settings

BB_PERIOD = 20
BB_STD = 2
INITIAL_CAPITAL = 100_000
START, END = "2018-01-01", None 
### END=None ⇒ today

tickers = ["GC=F", "AMD", "XLE"]

In [ ]:
# Download data

data = yf.download(tickers, start=START, end=END)["Close"]
data.head()

In [12]:
# Strategy Functions

def bollinger_backtest(price: pd.Series,
                       period:int = BB_PERIOD,
                       n_std:float = BB_STD,
                       initial:float = INITIAL_CAPITAL):

    df = price.to_frame("p")
    df["ma"]    = df["p"].rolling(period).mean()
    df["std"]   = df["p"].rolling(period).std()
    df["upper"] = df["ma"] + n_std*df["std"]
    df["lower"] = df["ma"] - n_std*df["std"]

    df = df.dropna()

    cash, shares, pos = initial, 0, 0
    trades, equity = [], []

    for t, r in df.iterrows():
        px = r["p"]

        ## BUY
        if pos == 0 and px < r["lower"]:
            shares = cash // px
            cash  -= shares*px
            pos = 1
            trades.append((t, "BUY", px, shares))

        ## SELL
        elif pos == 1 and px > r["ma"]:
            cash += shares*px
            trades.append((t, "SELL", px, shares))
            shares = 0
            pos = 0

        equity.append(cash + shares*px)

    df = df.iloc[-len(equity):].copy()
    df["equity"] = equity

    ret   = df["equity"].iloc[-1]/initial - 1
    dr    = df["equity"].pct_change().dropna()
    sharpe= np.sqrt(252)*dr.mean()/dr.std() if dr.std() else np.nan
    mdd   = ((df["equity"]/df["equity"].cummax())-1).min()

    return {"df":df, "trades":trades,
            "ret":ret, "sharpe":sharpe, "mdd":mdd}


In [ ]:
# Backtesting & Summary

results = []
for tk in tickers:
    s = data[tk].dropna()
    if len(s) < BB_PERIOD+5: continue
    r = bollinger_backtest(s)
    r["tk"] = tk
    results.append(r)

summary = pd.DataFrame({
        "Ticker":[r["tk"] for r in results],
        "Return_%":[round(r["ret"]*100,2) for r in results],
        "Sharpe":[round(r["sharpe"],2) for r in results],
        "MaxDD_%":[round(r["mdd"]*100,2) for r in results],
        "Trades":[len(r["trades"]) for r in results]
}).sort_values("Sharpe", ascending=False).reset_index(drop=True)

summary


In [ ]:
# Plots for each asset

for r in results:
    d = r["df"]; tk = r["tk"]
    fig, ax = plt.subplots()
    ax.plot(d.index, d["p"], label="Price")
    ax.plot(d.index, d["ma"], label="MA")
    ax.fill_between(d.index, d["upper"], d["lower"], alpha=.2, label="±2σ")

    for t, side, px, _ in r["trades"]:
        ax.scatter(t, px, marker="^" if side=="BUY" else "v",
                   color="g" if side=="BUY" else "r", s=70)

    ax.set_title(f"{tk}  |  Sharpe {r['sharpe']:.2f}  |  Return {r['ret']*100:.1f}%")
    ax.legend()
    plt.show()
